# NB-3: Ablation Study — 5 Forgetting Strategies
**Publication blockers PB-11 + PB-12**

Compares 5 forgetting strategies side-by-side:
1. **No-Forgetting** — memory grows unbounded (baseline)
2. **LRU** — least recently used (classic baseline)
3. **Importance** — forget least important first
4. **CA-Formula-Only** — consolidation formula, gate disabled (θ=0) — **new ablation**
5. **Consolidation-Aware (Ours)** — full formula + gate (θ=0.3) — **our contribution**

Also reports **False Forgetting Rate (FFR)** per strategy — fraction of evicted
memories that were not yet consolidated (PB-12).

**Time estimate:** ~30–60 min (5 strategies × real LLM QA via Groq)

## ⚠️ IMPORTANT: READ THIS FIRST

**For the next developer:** This notebook is part of a publication sprint handoff. Before running this notebook:

1. **Read the handoff document** at the root of the repository: `HANDOFF_DOCUMENT.md`
   - Look for the **"DEDICATED N3 SECTION FOR NEXT DEVELOPER"** section
   - Contains detailed instructions and common issues

2. **What you need to do:**
   - Run N3 (Ablation Study) with semantic similarity metrics
   - Takes 15 minutes (single seed 42) to 45 minutes (3 seeds for variance)
   - Generates JSON result file(s) with F1, semantic similarity, and False Forgetting Rate

3. **Key expectations:**
   - Every progress line MUST show `Sem=X.XXX` (semantic similarity)
   - If you don't see `Sem=`, something is wrong—restart kernel and check git pull
   - CA-Ours should win: highest F1, lowest FFR (0%)
   - Gate should prove necessary: CA-Formula-Only FFR > 0%, CA-Ours FFR = 0%

4. **When done:**
   - Download result JSON(s) to `/kaggle_results/`
   - Verify all files have `"avg_semantic_sim"` and `"semantic_sim_scores"` fields
   - Hand off to paper writer with metrics

**Questions?** Check HANDOFF_DOCUMENT.md section "DEDICATED N3 SECTION FOR NEXT DEVELOPER"

## Step 1 — Install & clone

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    print('Pulling latest changes from main...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                          capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('⚠ Pull warning:', result.stderr)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'csam_project'))
print(f'✓ Ready in {os.getcwd()}')

# Show latest commit
commit_result = subprocess.run(['git', 'log', '-1', '--oneline'], 
                              capture_output=True, text=True, cwd=REPO_DIR)
print(f'✓ Latest commit: {commit_result.stdout.strip()}')

## Step 2 — API key
**Kaggle:** Notebook → Settings → Secrets → `GROQ_API_KEY`

**Colab:** Left sidebar key icon → `GROQ_API_KEY`

In [ ]:
import threading, time

# ── Keep-alive: prevents Kaggle idle timeout (40-min limit) ──────────────────
# Background thread prints heartbeat every 20 min to keep the session alive.
#
# IMPORTANT: For safest execution use "Save Version → Run All" (batch mode).
# Batch mode has NO idle timeout, just a 9-hour hard limit. Checkpointing
# saves each completed strategy so an interrupt can be resumed.
def _keepalive():
    start = time.time()
    while True:
        time.sleep(1200)  # 20 minutes
        elapsed = (time.time() - start) / 60
        print(f"[keepalive] {elapsed:.0f} min elapsed — session active", flush=True)

_ka = threading.Thread(target=_keepalive, daemon=True)
_ka.start()
print("Keep-alive thread started (heartbeat every 20 min)")
print("Estimated runtime: ~30-60 min (well within 9-hour Kaggle limit)")

In [ ]:
import os

# ── Load ALL Groq keys (supports GROQ_API_KEY, GROQ_API_KEY_2 … GROQ_API_KEY_8) ──
def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')
print('Key rotation active — 429/400 errors auto-rotate to next key')

## Step 3 — Configure ablation run

In [ ]:
MODEL         = 'llama-3.1-8b-instant'
SEED          = 42
CONVERSATIONS = 5    # conversations to run (each generates QA pairs)
INTERACTIONS  = 50   # memory events per conversation (50=fast, 100=paper quality)
THRESHOLD     = 80   # forget trigger: evict when L2 > threshold memories

import os
safe_model = MODEL.replace('/', '_')
OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'

print(f'Model:         {MODEL}')
print(f'Conversations: {CONVERSATIONS}')
print(f'Interactions:  {INTERACTIONS}')
print(f'Threshold:     {THRESHOLD}')
print(f'Output:        {OUT_ABLATION}')
print(f'\nEstimated API calls: ~{CONVERSATIONS * 10 * 5} (5 strategies)')

### 📋 OUTPUT FILE NAMES

This cell configures the output filenames. After running, check that result files are created:

**Seed 42 (default):** `ablation_results_llama-3.1-8b-instant_s42.json`  
**Seed 123 (optional):** `ablation_results_llama-3.1-8b-instant_s123.json`  
**Seed 456 (optional):** `ablation_results_llama-3.1-8b-instant_s456.json`  

All files go to: `csam_project/evaluation/`

**To run multi-seed variance:** Uncomment the "ALTERNATE CONFIG" cells in Step 3, update SEED, and re-run Steps 4-5

In [ ]:
# ============================================================================
# ALTERNATE CONFIG 1: Seed 123 (same model, different seed)
# Uncomment this cell to override the above CONFIG and run with seed=123
# ============================================================================
# MODEL         = 'llama-3.1-8b-instant'
# SEED          = 123
# CONVERSATIONS = 5
# INTERACTIONS  = 50
# THRESHOLD     = 80
# 
# import os
# safe_model = MODEL.replace('/', '_')
# OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'
# 
# print(f'Config SEED=123:')
# print(f'Model:         {MODEL}')
# print(f'Seed:          {SEED}')
# print(f'Output:        {OUT_ABLATION}')

In [ ]:
# ============================================================================
# ALTERNATE CONFIG 2: Seed 456 (same model, different seed)
# Uncomment this cell to override the above CONFIG and run with seed=456
# ============================================================================
# MODEL         = 'llama-3.1-8b-instant'
# SEED          = 456
# CONVERSATIONS = 5
# INTERACTIONS  = 50
# THRESHOLD     = 80
# 
# import os
# safe_model = MODEL.replace('/', '_')
# OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'
# 
# print(f'Config SEED=456:')
# print(f'Model:         {MODEL}')
# print(f'Seed:          {SEED}')
# print(f'Output:        {OUT_ABLATION}')

In [ ]:
# ============================================================================
# ALTERNATE CONFIG 3: 70B Model (different model, seed 42)
# Uncomment this cell to override the above CONFIG and run with 70B model
# ============================================================================
# MODEL         = 'llama-3.3-70b-versatile'
# SEED          = 42
# CONVERSATIONS = 5
# INTERACTIONS  = 50
# THRESHOLD     = 80
# 
# import os
# safe_model = MODEL.replace('/', '_')
# OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'
# 
# print(f'Config 70B MODEL:')
# print(f'Model:         {MODEL}')
# print(f'Seed:          {SEED}')
# print(f'Output:        {OUT_ABLATION}')

In [ ]:
# ============================================================================
# ALTERNATE CONFIG 4: Scout 17B Model (different model, seed 42)
# Uncomment this cell to override the above CONFIG and run with Scout 17B model
# ============================================================================
# MODEL         = 'llama-4-scout-17b-16e-instruct'
# SEED          = 42
# CONVERSATIONS = 5
# INTERACTIONS  = 50
# THRESHOLD     = 80
# 
# import os
# safe_model = MODEL.replace('/', '_')
# OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'
# 
# print(f'Config Scout 17B MODEL:')
# print(f'Model:         {MODEL}')
# print(f'Seed:          {SEED}')
# print(f'Output:        {OUT_ABLATION}')

## Step 4 — Run ablation (5 strategies)
Runs all 5 strategies in sequence. Each uses the same conversations + QA pairs
so results are directly comparable. Progress is printed live.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.evaluation.run_ablation',
    '--conversations', str(CONVERSATIONS),
    '--interactions',  str(INTERACTIONS),
    '--threshold',     str(THRESHOLD),
    '--model',         MODEL,
    '--seed',          str(SEED),
    '--output',        OUT_ABLATION,
]
print('Starting ablation (5 strategies)...')
print('This may take 30-60 min depending on model speed.\n')
result = subprocess.run(cmd, capture_output=False, text=True)
print('\nAblation done' if result.returncode == 0 else 'Ablation FAILED — check output above')

## Step 5 — Display results table

In [ ]:
import json, os

if not os.path.exists(OUT_ABLATION):
    print(f'Output not found: {OUT_ABLATION}')
else:
    with open(OUT_ABLATION) as f:
        ab = json.load(f)

    print('=' * 80)
    print('ABLATION RESULTS — 5 Forgetting Strategies')
    print('=' * 80)
    print(f'{"Strategy":<35} {"F1":>7} {"Mem":>6} {"FFR":>7}')
    print('-' * 60)

    for r in ab.get('results', []):
        strategy = r['strategy']
        f1       = r.get('overall_f1', 0)
        mem      = r.get('memory_count', 0)
        ffr      = r.get('false_forgetting_rate', None)
        ffr_str  = f'{ffr:.3f}' if ffr is not None else 'N/A'
        marker   = ' ← OURS' if 'Consolidation-Aware (Ours)' in strategy else ''
        print(f'{strategy:<35} {f1:>7.4f} {mem:>6} {ffr_str:>7}{marker}')

    print('\nFFR = False Forgetting Rate (lower is better for CA-Ours)')
    print('Goal: CA-Ours FFR ≈ 0, all baselines FFR > 0')

    if ab.get('api_usage'):
        u = ab['api_usage']
        print(f'\nAPI: {u.get("total_requests","?")} requests, '
              f'{u.get("total_tokens",0):,} tokens')

## Step 6 — (Optional) Multi-seed variance run
Runs the ablation across 5 seeds to compute variance. Only run after the single-seed
ablation above confirms results look correct. Takes ~5× longer.

In [ ]:
# Uncomment to run multi-seed variance
# SEEDS = [42, 123, 456, 789, 1337]
# 
# for seed in SEEDS:
#     out = f'csam_project/evaluation/ablation_{safe_model}_s{seed}.json'
#     cmd = [
#         sys.executable, '-m', 'csam_project.evaluation.run_ablation',
#         '--conversations', str(CONVERSATIONS),
#         '--interactions',  str(INTERACTIONS),
#         '--threshold',     str(THRESHOLD),
#         '--model',         MODEL,
#         '--seed',          str(seed),
#         '--output',        out,
#     ]
#     print(f'Running seed={seed}...')
#     subprocess.run(cmd, capture_output=False, text=True)
print('Variance run commented out by default. Uncomment SEEDS block to enable.')

## Step 7 — Save results

In [ ]:
import shutil

if os.path.exists('/kaggle'):
    dest = os.path.join('/kaggle/working', os.path.basename(OUT_ABLATION))
    if os.path.exists(OUT_ABLATION):
        shutil.copy(OUT_ABLATION, dest)
        print(f'Saved to Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        if os.path.exists(OUT_ABLATION):
            files.download(OUT_ABLATION)
            print(f'Downloaded: {OUT_ABLATION}')
    except ImportError:
        print(f'File at: {OUT_ABLATION}')